In [2]:
import pandas as pd
import os
from pathlib import Path

Loading the datasets

In [65]:
notebook_dir = os.path.abspath("")
parent_dir = os.path.abspath("..")
data_folder = Path(parent_dir, "data")

# economic_indicators = ["gdp", "gdp_per_capita", "unemployment_rate"]
gdp_df = pd.read_csv(data_folder / "gdp.csv")
gdp_pc_df = pd.read_csv(data_folder / "gdp_per_capita.csv")
unemployment_df = pd.read_csv(data_folder / "unemployment_rate.csv")

# health_indicators
population_df = pd.read_csv(data_folder / "population.csv")
life_expectancy_df = pd.read_csv(data_folder / "life_expectancy.csv")
youth_mortality_df = pd.read_csv(data_folder / "youth_mortality.csv")
homicides_df = pd.read_csv(data_folder / "homicide_rate.csv")

# social_indicators
literacy_df = pd.read_csv(data_folder / "literacy_rate.csv")
human_rights_df = pd.read_csv(data_folder / "human_rights.csv")

# poverty_indicators
share_poverty_df = pd.read_csv(data_folder / "share_in_poverty.csv")
number_poverty_df = pd.read_csv(data_folder / "number_in_poverty.csv")

merging datasets based on the relatablity

In [66]:
economic_indicators_df = pd.merge(gdp_df, gdp_pc_df, on=["Entity", "Year"], how="outer").merge(unemployment_df, on=["Entity", "Year"], how="outer")
health_indicators_df = pd.merge(population_df, life_expectancy_df, left_on=["entity", "year"], right_on=["Entity", "Year"], how="outer").merge(youth_mortality_df, on=["Entity", "Year"], how="outer").merge(homicides_df, on=["Entity", "Year"], how="outer")
social_indicators_df = pd.merge(literacy_df, human_rights_df, on=["Entity", "Year"], how="outer")
poverty_indicators_df = pd.merge(share_poverty_df, number_poverty_df, on=["Country", "Year"], how="outer")


giving aliases to merged datasets

In [67]:
ei = economic_indicators_df
hi = health_indicators_df
si = social_indicators_df
pi = poverty_indicators_df

checking the max years in all datasets to get the mutual and latest year in all of them

In [68]:
list_of_data = [ei[ei.Year == ei.Year.max()].Year.unique(), 
                hi[hi.Year == hi.Year.max()].Year.unique(), 
                si[si.Year == si.Year.max()].Year.unique(),
                pi[pi.Year == pi.Year.max()].Year.unique()]

print(f"Latest common year in all datasets: {min(list_of_data)}")

Latest common year in all datasets: [2024.]


Renaming & Selecting (changing column names, selecting useful ones)

In [107]:
# economic indicators dataset
ei.rename({'Entity': 'country',
           'Year': 'year',
           'GDP': 'gdp',
           'GDP per capita': 'gdp_pc',
           'World region according to OWID': 'region',
           'Code': 'code',
           'Unemployment rate': 'unemployment_rate'}, axis=1, inplace=True)
ei = ei[['country', 'code', 'region', 'year', 'gdp', 'gdp_pc', 'unemployment_rate']]


# health indicators dataset
hi.rename({'entity': 'country',
           'population__sex_all__age_all__variant_estimates': 'total_population',
           'Life expectancy': 'life_expectancy',
           'Under-fifteen mortality rate': 'mortality_rate_u15',
           'Homicide rate per 100,000 population': 'homicide_rate_p100k',
           'World region according to OWID': 'region'}, axis=1, inplace=True)
hi = hi[['country', 'code', 'region', 'year', 'total_population', 'life_expectancy', 'mortality_rate_u15', 'homicide_rate_p100k']]


# social indicators dataset
si.rename({'Entity': 'country',
           'Year': 'year',
           'Literacy rate': 'literacy_rate',
           'Human Rights Index': 'human_rights_index',
           'Code_x': 'code',
           'World region according to OWID': 'region'}, axis=1, inplace=True)
si = si[['country', 'code', 'region', 'year', 'literacy_rate', 'human_rights_index']]



# poverty_indicators_df
pi.rename({'Country': 'country',
           'Year': 'year',
           'Share below $3 a day': 'poverty_share',
           'Number below $3 a day': 'poverty_number'}, axis=1, inplace=True)
pi = pi[['country', 'year', 'poverty_share', 'poverty_number']]



Selecting the data only between the span of 10 years from 2015 - 2024

In [108]:
ei = ei[(ei.year > 2014) & (ei.year < 2025)]
hi = hi[(hi.year > 2014) & (hi.year < 2025)]
si = si[(si.year > 2014) & (si.year < 2025)]
pi = pi[(pi.year > 2014) & (pi.year < 2025)]

In [132]:
ei.info()

<class 'pandas.DataFrame'>
Index: 2253 entries, 24 to 12598
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   country            2253 non-null   str    
 1   code               1991 non-null   str    
 2   region             1984 non-null   str    
 3   year               2253 non-null   int64  
 4   gdp                2203 non-null   float64
 5   gdp_pc             2114 non-null   float64
 6   unemployment_rate  1991 non-null   float64
dtypes: float64(3), int64(1), str(3)
memory usage: 140.8 KB
